# 🧪 Laboratório 3 — Regressão espacial: SAR, SEM e SDM num mundo onde a verdade é conhecida

**Minicurso LaPSEE · UNESP Ilha Solteira**

Nos dados reais nunca sabemos se o estimador acertou — não conhecemos a verdade. Hoje vocês
trabalham no único lugar onde dá para **auditar** um estimador: um mundo sintético em que
**nós escolhemos** $\rho$, $\beta$ e o mecanismo, geramos os dados, e depois fingimos não
saber. Se o método não recuperar a verdade *aqui*, não merece confiança *lá fora*.

Vocês são a consultoria de estatística do operador da rede. As perguntas do dia são as
quatro da aula (Parte V):

1. O que move o preço, controlando por covariáveis? → $\beta$
2. Se intervenho na barra $j$, o que acontece nas *outras*? → efeito indireto
3. O agrupamento vem da **rede** ou do **clima comum**? → SAR vs. SEM (bateria LM)
4. O vizinho entra por $y$, pelo erro, ou por $X$? → SAR / SEM / SDM

## Regras do jogo

1. Pode trabalhar em equipe.
2. Blocos `# ╔══ COMPLETAR` são de vocês; a **VERIFICAÇÃO** (`assert`) audita o resultado.
3. As perguntas 📝 compõem o **entregável**: um parecer técnico de 1 página.



---
## 0 · Setup: a cidade de 64 barras (entregue completo)

Nosso mundo é uma cidade-grade $8\times 8$ — o mesmo *lattice* do exemplo trabalhado da
aula. Duas covariáveis com história:

- **demanda** $D_i$ [MW]: heterogênea lote a lote (sorteio de zoneamento), com um leve
  adensamento em direção ao centro;
- **capacidade** $E_i$ [MW]: heterogênea, com um corredor de geração na borda **leste**.

A variável de interesse $y$ será um índice de preço local [\$/MWh], que **nós mesmos**
geraremos nos blocos seguintes.

In [ ]:
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from libpysal.weights import lat2W
from esda.moran import Moran, Moran_Local
import spreg

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": False, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
AZUL, VERMELHO, CINZA = "#1F3A5F", "#C8102E", "#a0a0a0"

LADO = 8
N = LADO * LADO                                    # 64 barras

# --- as covariáveis da cidade (seed fixa: todos têm a MESMA cidade) -----------
rng_cidade = np.random.default_rng(2026)
xx, yy = np.meshgrid(np.arange(LADO), np.arange(LADO))
dist_centro = np.sqrt((xx - 3.5) ** 2 + (yy - 3.5) ** 2)
demanda = (45 + rng_cidade.normal(0, 12, (LADO, LADO)) - 2.0 * dist_centro).ravel()
capacidade = (35 + rng_cidade.normal(0, 10, (LADO, LADO)) + 6 * (xx >= 5)).ravel()
X = np.column_stack([demanda, capacidade])
NOMES_X = ["demanda", "capacidade"]


def mapa_calor(vec, titulo="", ax=None, cmap="plasma", vmin=None, vmax=None):
    "Heatmap 8×8 de um vetor de 64 valores (a 'planta' da cidade)."
    solo = ax is None
    if solo:
        fig, ax = plt.subplots(figsize=(4.6, 4.2))
    im = ax.imshow(np.asarray(vec).reshape(LADO, LADO), cmap=cmap,
                   vmin=vmin, vmax=vmax)
    ax.set_title(titulo, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)
    if solo:
        plt.tight_layout(); plt.show()


fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))
mapa_calor(demanda, "demanda $D_i$ [MW] — adensa no centro", ax=axes[0])
mapa_calor(capacidade, "capacidade $E_i$ [MW] — corredor a leste", ax=axes[1], cmap="viridis")
plt.tight_layout(); plt.show()
print("Cidade pronta:", N, "barras")

---
## Bloco A · A fábrica de mundos

### A.1 — A matriz $W$ da cidade

Vizinhança *rook* (contiguidade de borda) na grade, padronizada por linhas — exatamente a
$W$ do exemplo da aula. `libpysal.weights.lat2W(L, L, rook=True)` constrói o objeto; o
atributo `.transform = "r"` padroniza.

In [ ]:
w = lat2W(LADO, LADO, rook=True)
w.transform = "r"
W = np.array(w.full()[0])
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert W.shape == (N, N), "W deve ser 64×64"
assert np.allclose(W.sum(axis=1), 1.0), "padronizada por linhas: cada linha soma 1"
n_viz = (W > 0).sum(axis=1)
assert n_viz[0] == 2 and n_viz[3 * LADO + 3] == 4, \
    "canto deve ter 2 vizinhos; interior, 4 (rook)"
print(f"✅ VERIFICAÇÃO OK — vizinhos: canto = {n_viz[0]}, interior = {n_viz[3*LADO+3]}")

### A.2 — O gerador de mundos SAR

A forma reduzida da aula é a receita do DGP:

$$\vec y \;=\; (\mathbb I - \rho W)^{-1}\,\bigl(\beta_0 + X\beta + \varepsilon\bigr),
\qquad \varepsilon \sim \mathcal N(0, \sigma^2).$$

Implementem `gerar_sar(...)` exatamente assim — o $(\mathbb I-\rho W)^{-1}$ é a "matriz de
alcance" que propaga cada choque pela rede inteira, com decaimento.

In [ ]:
def gerar_sar(rho0, beta0, beta_dem, beta_cap, sigma, seed):
    "Gera y do DGP SAR pela forma reduzida (a receita da aula)."
    g = np.random.default_rng(seed)
    # ╔══ COMPLETAR ══ dica: eps = g.normal(0, sigma, N);
    # ║   A = np.linalg.inv(np.eye(N) - rho0 * W);
    # ║   y = A @ (beta0 + X @ np.array([beta_dem, beta_cap]) + eps)
    # ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
    raise NotImplementedError("Bloco a completar — vejam a dica acima")
    return y


# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
g = np.random.default_rng(72)
eps_demo = g.normal(0, 2.0, N)                       # MESMO choque nos dois mundos
y_sem_rho = 5.0 + X @ np.array([0.30, -0.15]) + eps_demo
y_com_rho = np.linalg.inv(np.eye(N) - 0.60 * W) @ (5.0 + X @ np.array([0.30, -0.15]) + eps_demo)
y_teste = gerar_sar(0.60, 5.0, 0.30, -0.15, 2.0, seed=72)
assert np.allclose(y_teste, y_com_rho), "a função deve reproduzir a forma reduzida"
I0 = Moran(y_sem_rho, w, permutations=99).I
I6 = Moran(y_teste, w, permutations=99).I
assert I6 > 0.4 and I0 < 0.3, "com rho=0.6 o Moran deve disparar"
print(f"✅ VERIFICAÇÃO OK — Moran I: rho=0 → {I0:.3f} | rho=0.6 → {I6:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))
v = max(abs(y_sem_rho - y_sem_rho.mean()).max(), abs(y_teste - y_teste.mean()).max())
mapa_calor(y_sem_rho - y_sem_rho.mean(), f"ρ = 0  (I = {I0:.2f})", ax=axes[0],
           cmap="RdBu_r", vmin=-v, vmax=v)
mapa_calor(y_teste - y_teste.mean(), f"ρ = 0.6  (I = {I6:.2f})", ax=axes[1],
           cmap="RdBu_r", vmin=-v, vmax=v)
fig.suptitle("O MESMO choque ε, com e sem rede: ρ transforma sal-e-pimenta em manchas")
plt.tight_layout(); plt.show()

💡 Comente com base nos resultados o efeito de  $(\mathbb I - \rho W)^{-1}$. O $\rho$ 


📝 **Pergunta A:** no mapa com $\rho = 0.6$, ainda dá para enxergar o corredor de capacidade
do leste (que *baixa* o preço, $\beta_{cap} < 0$)? O que o $\rho$ fez com as fronteiras
nítidas dos efeitos de $X$?

> *Resposta de vocês aqui:* ______




---
## Bloco B · O julgamento do OLS

### B.1 — Uma réplica do mundo da aula

O mundo da aula: $\rho_0 = 0.60$, $\beta_0 = 5$, $\beta_{dem} = +0.30$,
$\beta_{cap} = -0.15$, $\sigma = 2$. Gerem a réplica nº 72, ajustem **OLS** (modelo errado,
sem termo espacial) e **SAR por máxima verossimilhança**, e montem a tabela
*estimativa vs. verdade*.

In [ ]:
VERDADE = {"rho": 0.60, "const": 5.0, "demanda": 0.30, "capacidade": -0.15}
y72 = gerar_sar(0.60, 5.0, 0.30, -0.15, 2.0, seed=72)

# ╔══ COMPLETAR ══ dica: ols = spreg.OLS(y72.reshape(-1,1), X, w=w, spat_diag=True,
# ║   name_x=NOMES_X); sar = spreg.ML_Lag(y72.reshape(-1,1), X, w=w, method="full",
# ║   name_x=NOMES_X). Os betas saem de np.ravel(modelo.betas); no SAR, rho fica ao final.
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert abs(float(sar.rho) - 0.60) < 0.10, "o ML deveria recuperar rho ≈ 0.6"
assert abs(b_sar[1] - 0.30) < 0.08 and abs(b_sar[2] + 0.15) < 0.08, \
    "o ML deveria recuperar os betas"
assert abs(b_ols[0] - 5.0) > 4.0, "o intercepto do OLS deveria estar MUITO longe de 5"
print("✅ VERIFICAÇÃO OK")
display(tabela.round(3))

💡 **Leiam a tabela como na aula:**

- **ML SAR acerta tudo** (dentro do erro amostral): $\hat\rho \approx 0.61$,
  $\hat\beta_{dem} \approx 0.30$, $\hat\beta_{cap} \approx -0.15$.
- **O OLS, sem ter onde pôr o $\rho$, despeja o feedback no intercepto** (≈ 15 em vez
  de 5!) e infla os betas. Quem lesse o OLS diria que a demanda "vale" o dobro.
- Detalhe honesto: esta é a réplica nº 72 — uma réplica *típica*. Réplicas individuais podem
  errar mais (o seed 2026, por exemplo, dá $\hat\beta_{dem} = 0.46$!). Estimativa pontual
  sem distribuição amostral é meia-verdade — e é exatamente o que o B.2 resolve.

### B.2 — O Monte Carlo: viés que não encolhe (código entregue)

Duas perguntas que só um Monte Carlo responde: o "OLS ingênuo" (regredir $y$ em $Wy$ e $X$
por mínimos quadrados) é viesado? E o viés desaparece com mais dados? Repetimos o mundo
SAR puro ($\rho_0 = 0.5$) centenas de vezes, em $n = 64$ e em $n = 256$.

In [ ]:
def mc_rho(L, reps, rho0=0.5, ml=False, seed=7):
    "Distribuição amostral de rho_hat: OLS ingênuo (lstsq com Wy) ou ML."
    w_ = lat2W(L, L, rook=True); w_.transform = "r"
    W_ = np.array(w_.full()[0]); nn = L * L
    A_ = np.linalg.inv(np.eye(nn) - rho0 * W_)
    g = np.random.default_rng(seed)
    saida = []
    for _ in range(reps):
        yv = A_ @ g.normal(0, 1, nn)                 # SAR puro: y = (I-ρW)⁻¹ ε
        x_irrel = g.normal(0, 1, (nn, 1))            # regressor irrelevante (β = 0)
        if ml:
            m = spreg.ML_Lag(yv.reshape(-1, 1), x_irrel, w=w_, method="full")
            saida.append(float(m.rho))
        else:
            Wy = W_ @ yv
            Z = np.column_stack([np.ones(nn), Wy, x_irrel])
            saida.append(np.linalg.lstsq(Z, yv, rcond=None)[0][1])
    return np.array(saida)


r_ols_64 = mc_rho(8, reps=200)
r_ols_256 = mc_rho(16, reps=200)
r_ml_64 = mc_rho(8, reps=120, ml=True)

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.hist(r_ols_64, bins=24, alpha=0.55, color=VERMELHO, density=True,
        label=f"OLS ingênuo, n=64 (média {r_ols_64.mean():.2f})")
ax.hist(r_ols_256, bins=24, alpha=0.55, color="#e58a5f", density=True,
        label=f"OLS ingênuo, n=256 (média {r_ols_256.mean():.2f})")
ax.hist(r_ml_64, bins=24, alpha=0.6, color=AZUL, density=True,
        label=f"ML, n=64 (média {r_ml_64.mean():.2f})")
ax.axvline(0.5, color="k", lw=2, ls="--", label="verdade ρ₀ = 0.5")
ax.set_xlabel(r"$\hat\rho$"); ax.set_ylabel("densidade"); ax.grid(alpha=0.3)
ax.set_title("Monte Carlo: o viés do OLS NÃO encolhe com mais dados; o ML é centrado")
ax.legend(); plt.tight_layout(); plt.show()

assert r_ols_64.mean() > 0.7 and r_ols_256.mean() >= r_ols_64.mean() - 0.02, \
    "o viés do OLS deveria persistir (ou crescer) em n=256"
assert abs(r_ml_64.mean() - 0.5) < 0.05, "o ML deveria estar centrado na verdade"
print(f"✅ VERIFICAÇÃO OK — viés OLS: n=64 → {r_ols_64.mean()-0.5:+.2f}, "
      f"n=256 → {r_ols_256.mean()-0.5:+.2f} | ML: {r_ml_64.mean()-0.5:+.2f}")

💡 **Resultado contraintuitivo nº1 — mais dados não salvam o OLS.** O $\hat\rho$ ingênuo
fica em ≈ 0.8 (verdade: 0.5) e em $n = 256$ o viés **não encolhe**: é *inconsistência*, não
amostra pequena. O motivo é o da aula: $Wy$ é endógeno — cada $y_j$ vizinho contém o meu
próprio $\varepsilon_i$ refletido pela rede, e o numerador do OLS "credita" essa
retroalimentação ao $\rho$. O Jacobiano $\log|\mathbb I - \rho W|$ da máxima verossimilhança
é exatamente o termo que paga essa conta.

💡 **Resultado contraintuitivo nº2 — o intercepto mente primeiro.** Antes de olhar betas,
olhem o intercepto do OLS: 3× a verdade. Num modelo espacial mal especificado, o nível
"que sobra" é o primeiro lugar onde o $\rho$ omitido se esconde.

📝 **Pergunta B:** um colega propõe: *"com 10.000 barras o problema do OLS desaparece"*.
Respondam em 2 frases usando o histograma.

> *Resposta de vocês aqui:* ______

---
## Bloco C · Rede ou clima? 


A pergunta nº 3 da aula: o agrupamento espacial dos preços vem da **rede** (os preços
vizinhos se puxam: SAR) ou de um **choque regional comum** — clima, combustível — escondido
no erro (SEM)? A resposta muda a política: se é rede, intervenções *propagam*; se é clima,
reforçar a rede não resolve nada.

O monitor de mercado entrega dois conjuntos de dados **sem etiqueta** — um gerado por cada
mecanismo (a célula abaixo os gera; finjam que não viram os parâmetros 🙈).

In [ ]:
# --- o monitor gera os dois mundos (NÃO espiem os mecanismos... ainda) --------
A_de = lambda r: np.linalg.inv(np.eye(N) - r * W)
g = np.random.default_rng(11)
mundo_A = A_de(0.5) @ (5.0 + X @ np.array([0.30, -0.15]) + g.normal(0, 2.0, N))
g = np.random.default_rng(57)
mundo_B = 5.0 + X @ np.array([0.30, -0.15]) \
        + np.linalg.inv(np.eye(N) - 0.7 * W) @ g.normal(0, 3.0, N)

I_A = Moran(mundo_A, w, permutations=199)
I_B = Moran(mundo_B, w, permutations=199)
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))
mapa_calor(mundo_A, f"mundo A — Moran I = {I_A.I:.2f} (p = {I_A.p_sim})", ax=axes[0])
mapa_calor(mundo_B, f"mundo B — Moran I = {I_B.I:.2f} (p = {I_B.p_sim})", ax=axes[1])
plt.tight_layout(); plt.show()
print("Os dois têm autocorrelação significativa. O Moran DETECTA — mas não ATRIBUI.")

### C.1 — A bateria LM decide

Rodem `spreg.OLS(..., spat_diag=True)` em cada mundo e extraiam os **LM robustos** (a única
parte da bateria que separa os mecanismos quando ambos os testes simples disparam). Depois
preencham o veredito.

| Padrão dos LM robustos | Veredito |
|---|---|
| Robust LM (lag) rejeita, Robust LM (error) não | **rede** (SAR) |
| Robust LM (error) rejeita, Robust LM (lag) não | **clima** (SEM) |

In [ ]:
def lm_robustos(yv):
    "Extrai os p-valores dos LM robustos do summary do OLS."
    o = spreg.OLS(yv.reshape(-1, 1), X, w=w, spat_diag=True, name_x=NOMES_X)
    pares = re.findall(r"(Robust LM \((?:lag|error)\))\s+\d+\s+([\d.]+)\s+([\d.]+)",
                       o.summary)
    return {t: float(p) for t, _e, p in pares}


# ╔══ COMPLETAR ══ dica: chamem lm_robustos em cada mundo, leiam os p-valores impressos
# ║   e preencham o dicionário `veredito` com "rede" ou "clima" para cada mundo
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert p_A["Robust LM (lag)"] < 0.05 < p_A["Robust LM (error)"], \
    "no mundo A, o lag robusto deveria rejeitar e o error robusto, não"
assert p_B["Robust LM (error)"] < 0.05 < p_B["Robust LM (lag)"], \
    "no mundo B, o error robusto deveria rejeitar e o lag robusto, não"
assert veredito == {"mundo_A": "rede", "mundo_B": "clima"}, "veredito incorreto!"
print("✅ VERIFICAÇÃO OK — mundo A = REDE (SAR, ρ₀=0.5) | mundo B = CLIMA (SEM, λ₀=0.7)")

💡 Comente os resultados.




📝 **Pergunta C:** para cada mundo, digam em 1 frase qual seria a recomendação de política
errada se o analista tivesse trocado os diagnósticos (tratado rede como clima e vice-versa).

> *Resposta de vocês aqui:* ______


---
## Bloco D · SDM: quando o vizinho entra por $X$

Terceiro mecanismo da aula: o **covariável do vizinho** move o meu $y$ — um gerador novo na
barra ao lado entra no meu preço pela coluna da PTDF. O DGP agora tem $\theta$:

$$\vec y = \rho W \vec y + X\beta + W\!X\,\theta + \varepsilon,
\qquad \theta_{dem} = +0.25 \;(\text{a demanda do vizinho também me encarece}).$$

O monitor gera o mundo SDM; vocês ajustam **SAR (mal especificado)** e **SDM** no mesmo dado
e comparam os **efeitos totais** — a grandeza que vai para a decisão. No `spreg`, o SDM é o
próprio `ML_Lag` com `slx_lags=1` (ele acrescenta as colunas $W\!X$).

In [ ]:
g = np.random.default_rng(19)
y_sdm = A_de(0.5) @ (5.0 + X @ np.array([0.30, -0.15])
                     + (W @ X) @ np.array([0.25, 0.0]) + g.normal(0, 2.0, N))
VERDADE_SDM = {"rho": 0.5, "beta_dem": 0.30, "theta_dem": 0.25}
total_dem_verdade = (0.30 + 0.25) / (1 - 0.5)          # = 1.10 $/MWh por MW

# ╔══ COMPLETAR ══ dica: sar_mal = spreg.ML_Lag(y, X, w=w, method="full");
# ║   sdm = spreg.ML_Lag(y, X, w=w, method="full", slx_lags=1).
# ║   Em sdm.betas a ordem é [const, demanda, capacidade, W_demanda, W_capacidade, rho].
# ║   Efeito TOTAL da demanda: SAR → beta_dem/(1-rho); SDM → (beta_dem+theta_dem)/(1-rho)
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert 0.35 < rho_s < 0.65 and 0.10 < b_s[3] < 0.40, \
    "o SDM deveria recuperar rho ≈ 0.5 e theta_dem ≈ 0.25"
assert abs(total_sdm - total_dem_verdade) < abs(total_sar - total_dem_verdade), \
    "o efeito total do SDM deveria estar mais perto da verdade que o do SAR"
comp = pd.DataFrame({
    "modelo": ["verdade", "SDM (correto)", "SAR (mal especificado)"],
    "rho": [0.5, rho_s, rho_m],
    "beta_dem": [0.30, b_s[1], b_m[1]],
    "theta_dem (W·dem)": [0.25, b_s[3], np.nan],
    "efeito TOTAL da demanda": [total_dem_verdade, total_sdm, total_sar],
}).round(3)
print("✅ VERIFICAÇÃO OK")
display(comp)
erro_pct = 100 * (total_sar - total_dem_verdade) / total_dem_verdade
print(f"O SAR erra o efeito total em {erro_pct:+.0f}% — e parece um ajuste perfeitamente saudável.")

💡 **Resultado contraintuitivo nº4 — o modelo errado parece saudável.** O SAR mal
especificado entrega um $\hat\rho$ plausível, betas significativos, bom ajuste... e um
efeito total ~15% longe da verdade, porque não tem onde pôr o $\theta$: parte do efeito do
vizinho vaza para $\hat\beta$ e $\hat\rho$. Escolher o modelo é uma **afirmação sobre o
mecanismo** — não um concurso de $R^2$. (LeSage & Pace 2008: na dúvida, **o SDM é o padrão
seguro** — continua consistente para os efeitos sob as duas má-especificações mais comuns.)



📝 **Pergunta D:** suponham que a decisão fosse "onde uma campanha de eficiência (reduzir
demanda) baixa mais o índice?". Expliquem em 2 frases por que a resposta do SAR e a do SDM
divergem mesmo com betas parecidos.

> *Resposta de vocês aqui:* ______



---
## Bloco E · Mapas de influência: da regressão à decisão de sítio

### E.1 — LISA: onde vivem os clusters (código entregue)

Os dois gráficos canônicos da econometria espacial aplicada, sobre a réplica nº 72 do
Bloco B: o *Moran scatterplot* e o **mapa de clusters LISA**.

In [ ]:
lisa = Moran_Local(y72, w, permutations=999, seed=2026)
sig = lisa.p_sim < 0.05
rotulo = np.zeros(N)                                # 0=NS, 1=HH, 2=LL, 3=HL/LH
rotulo[sig & (lisa.q == 1)] = 1
rotulo[sig & (lisa.q == 3)] = 2
rotulo[sig & ((lisa.q == 2) | (lisa.q == 4))] = 3

z = (y72 - y72.mean()) / y72.std()
Wz = W @ z
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].scatter(z, Wz, c=AZUL, s=35)
pend = np.polyfit(z, Wz, 1)[0]
xs = np.linspace(z.min(), z.max(), 40)
axes[0].plot(xs, pend * xs, c=VERMELHO, label=f"inclinação = I = {pend:.2f}")
axes[0].axhline(0, c="k", lw=0.6); axes[0].axvline(0, c="k", lw=0.6)
axes[0].set_xlabel("z (índice padronizado)"); axes[0].set_ylabel("Wz (média dos vizinhos)")
axes[0].set_title("Moran scatterplot"); axes[0].legend(); axes[0].grid(alpha=0.3)

from matplotlib.colors import ListedColormap
cm = ListedColormap(["#e7e7e7", VERMELHO, AZUL, "#f4a261"])
im = axes[1].imshow(rotulo.reshape(LADO, LADO), cmap=cm, vmin=0, vmax=3)
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].set_title("LISA: vermelho HH (bolsão caro) · azul LL (bolsão barato) · laranja outliers")
plt.tight_layout(); plt.show()
print(f"Clusters: HH = {(rotulo==1).sum()} barras | LL = {(rotulo==2).sum()} | "
      f"outliers = {(rotulo==3).sum()}")

### E.2 — O mapa de influência de uma intervenção

A coluna $j$ de $S_{cap} = \beta_{cap}\,(\mathbb I - \rho W)^{-1}$ responde à pergunta de
decisão: *"se eu adiciono capacidade na barra $j$, quanto cai o índice em **cada** barra da
cidade?"*. Construam as colunas para duas candidatas — **centro** (barra 27) e **canto**
(barra 0) — usando o $\hat\rho$ e o $\hat\beta_{cap}$ do SAR do Bloco B, e comparem as somas.

In [ ]:
j_centro, j_canto = 3 * LADO + 3, 0

# ╔══ COMPLETAR ══ dica: A_hat = np.linalg.inv(np.eye(N) - rho_hat*W) com
# ║   rho_hat = float(sar.rho); beta_cap_hat = np.ravel(sar.betas)[2];
# ║   col_centro = beta_cap_hat * A_hat[:, j_centro] (idem canto); somas com .sum()
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert np.allclose(A_hat.sum(axis=1), A_hat.sum(axis=1)[0]), \
    "as somas de LINHA de (I-ρW)⁻¹ devem ser todas iguais (W padronizada)"
assert abs(soma_centro) > abs(soma_canto), \
    "o efeito sistêmico do CENTRO deve superar o do canto"
print(f"✅ VERIFICAÇÃO OK — efeito sistêmico de +1 MW de capacidade: "
      f"centro {soma_centro:+.3f} vs canto {soma_canto:+.3f} $/MWh "
      f"({abs(soma_centro/soma_canto):.2f}×)")

v = max(abs(col_centro).max(), abs(col_canto).max())
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))
mapa_calor(col_centro, f"+1 MW no CENTRO → Σ = {soma_centro:+.2f}", ax=axes[0],
           cmap="RdBu_r", vmin=-v, vmax=v)
mapa_calor(col_canto, f"+1 MW no CANTO → Σ = {soma_canto:+.2f}", ax=axes[1],
           cmap="RdBu_r", vmin=-v, vmax=v)
fig.suptitle("Mapas de influência: a MESMA intervenção, alcances diferentes")
plt.tight_layout(); plt.show()

💡 comente os resultados.


📝 **Pergunta E:** com $\hat\beta_{cap} < 0$, um subsídio de 10 MW de capacidade no centro
reduz o índice da cidade em quanto, somado sobre as 64 barras? E se o objetivo fosse
proteger *uma barra específica* da periferia, a resposta mudaria?

> *Resposta de vocês aqui:* ______
